<a href="https://colab.research.google.com/github/mdonbruce/AspNetDocs/blob/master/Lab2_Library_Polymorphism_Instructor_EXECUTED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 — Library Catalog: Instructor EXECUTED

Complete solution with example outputs.

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path
DATA_FILE = Path("lab2_library_checkouts.csv")
if not DATA_FILE.exists():
    DATA_FILE = Path(r"/mnt/data/module3_inheritance_polymorphism/lab2_library_checkouts.csv")
print("Using:", DATA_FILE.resolve())

df = pd.read_csv(DATA_FILE)
df.head()

Using: /mnt/data/module3_inheritance_polymorphism/lab2_library_checkouts.csv


,item_id,item_type,genre,length_units,publication_year,member_type,checkout_days,late_days,damage_flag
0,100001,Book,Nonfiction,636,1979,Public,11,1,N
1,100002,Book,Kids,814,1966,Public,9,1,N
2,100003,Book,SciFi,281,2023,Faculty,32,3,N
3,100004,Book,History,215,1991,Faculty,12,0,N
4,100005,Ebook,Fiction,781,2020,Public,20,0,N


In [ ]:
print(df.shape)
df.isna().sum().head()

(40000, 9)


item_id             0
item_type           0
genre               0
length_units        0
publication_year    0
dtype: int64

In [ ]:
class MediaItem:
    DAMAGE_FEE = 12.0
    def __init__(self, genre, member_type, late_days, damage_flag):
        self.genre = genre
        self.member_type = member_type
        self.late_days = int(late_days)
        self.damage_flag = damage_flag

    def loan_period_days(self):
        raise NotImplementedError

    def late_fee_per_day(self):
        raise NotImplementedError

    def total_fee(self):
        fee = self.late_days * self.late_fee_per_day()
        if self.damage_flag == 'Y':
            fee += self.DAMAGE_FEE
        return round(fee, 2)

class Book(MediaItem):
    def loan_period_days(self):
        return 21 if self.member_type != 'Public' else 14
    def late_fee_per_day(self):
        return 0.25

class Ebook(MediaItem):
    def loan_period_days(self):
        return 14
    def late_fee_per_day(self):
        return 0.10

class Audiobook(MediaItem):
    def loan_period_days(self):
        return 14
    def late_fee_per_day(self):
        return 0.20

class DVD(MediaItem):
    def loan_period_days(self):
        return 7
    def late_fee_per_day(self):
        return 0.75


In [ ]:
def make_item(row):
    t = row['item_type']
    cls = {'Book':Book,'Ebook':Ebook,'Audiobook':Audiobook,'DVD':DVD}.get(t, MediaItem)
    return cls(row['genre'], row['member_type'], row['late_days'], row['damage_flag'])


In [ ]:
sample = df.sample(5000, random_state=42)
items = [make_item(r) for _, r in sample.iterrows()]
fees = np.array([it.total_fee() for it in items])
fees.mean(), fees.max()

(0.6708299999999999, 17.25)

In [ ]:
sample.assign(fee=fees).groupby('item_type')['fee'].agg(['mean','median','max']).sort_values('mean', ascending=False)

,mean,median,max
item_type,,,
DVD,1.101974,0.0,17.25
Book,0.670113,0.0,13.75
Audiobook,0.622527,0.0,12.80
Ebook,0.509321,0.0,12.40
